In [6]:
import sys
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.functions import current_timestamp, lit

# --- 1. CONFIGURATION ---
catalog_name   = "glue_catalog"
database_name  = "processdb_iceberg"
source_db      = "landingdb"
source_table   = "formulaone-circuits_csv"
warehouse_path = "s3://formulaone-glue/process_iceberg/"

file_name_audit = source_table.replace("formulaone-", "").replace("_", ".")
# We use backticks in the path to ensure Spark separates Catalog from Database
target_table_path = f"`{catalog_name}`.`{database_name}`.`circuits`"
# target_table_path = f"`{database_name}`.`circuits`"

# --- 2. INITIALIZATION (The "Clean" Way) ---
conf = SparkConf()
conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
conf.set(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog")
conf.set(f"spark.sql.catalog.{catalog_name}.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
conf.set(f"spark.sql.catalog.{catalog_name}.warehouse", warehouse_path)
conf.set(f"spark.sql.catalog.{catalog_name}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
conf.set(f"spark.sql.catalog.{catalog_name}.type", "glue")

sc = SparkContext.getOrCreate(conf=conf)
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# --- 3. CREATE NAMESPACE (The Fix for 'Invalid Name') ---
print(f"Ensuring database {database_name} exists...")
# Backticks are the key here to stop the 'dot' error
# spark.sql(f"CREATE NAMESPACE IF NOT EXISTS `{catalog_name}`.`{database_name}`")
# spark.sql(f"CREATE DATABASE IF NOT EXISTS `{database_name}`")

# --- 4. READ & TRANSFORM ---
print(f"Reading from {source_db}.{source_table}...")
circuits_dynamic_frame = glueContext.create_dynamic_frame.from_catalog(
    database=source_db, 
    table_name=source_table
)

df = circuits_dynamic_frame.toDF()

rename_map = {
    "circuitid": "circuit_id", 
    "circuitRef": "circuit_ref",
    "lat": "latitude", 
    "lng": "longitude", 
    "alt": "altitude"
}

for old, new in rename_map.items():
    df = df.withColumnRenamed(old, new)

circuits_df = df.withColumn("load_timestamp", current_timestamp()) \
                .withColumn("file_name", lit(file_name_audit)) \
                .drop("url")

# --- 5. WRITE TO ICEBERG ---
print(f"Writing to Iceberg table: {target_table_path}...")

# .using("iceberg") ensures Spark uses the V2 engine instead of V1 Hive
circuits_df.writeTo(target_table_path) \
    .tableProperty("format-version", "2") \
    .tableProperty("write.format.default", "parquet") \
    .using("iceberg") \
    .createOrReplace()

# --- 6. VERIFICATION ---
print("\n--- Process Complete! ---")
spark.table(target_table_path).show(5, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
An error occurred while calling o185.getCatalogSource.
: com.amazonaws.services.glue.model.EntityNotFoundException: Entity Not Found (Service: AWSGlue; Status Code: 400; Error Code: EntityNotFoundException; Request ID: ea78d784-6fd5-47bd-86d6-16e5539fb7da; Proxy: null)
	at com.amazonaws.http.AmazonHttpClient$RequestExecutor.handleErrorResponse(AmazonHttpClient.java:1879)
	at com.amazonaws.http.AmazonHttpClient$RequestExecutor.handleServiceErrorResponse(AmazonHttpClient.java:1418)
	at com.amazonaws.http.AmazonHttpClient$RequestExecutor.executeOneRequest(AmazonHttpClient.java:1387)
	at com.amazonaws.http.AmazonHttpClient$RequestExecutor.executeHelper(AmazonHttpClient.java:1157)
	at com.amazonaws.http.AmazonHttpClient$RequestExecutor.doExecute(AmazonHttpClient.java:814)
	at com.amazonaws.http.AmazonHttpClient$RequestExecutor.executeWithTimer(AmazonHttpClient.java:781)
	at com.amazonaws.http.AmazonHttpClient$RequestExecutor.execute(AmazonHttpClient.java:755)
	at c